# AeroScapes Aerial Semantic Segmentation Experiment
### Model A (Baseline: No Skips) vs. Model B (Controlled: Skip Fusion)

This notebook runs the complete end-to-end reproducible experiment on **Google Colab (free-tier NVIDIA Tesla T4 GPU)**.

- **Dataset**: AeroScapes (3,269 drone-view scenes, 720p, 12 classes including background)
- **Architecture A**: Compact 4-stage encoder-decoder, bottleneck only, **no skip connections** (6.46M parameters)
- **Architecture B**: Identical to Model A with **skip fusion via feature concatenation** into the decoder (7.24M parameters)
- **Controlled Environment**: Identical seed (42), resolution (384x256), batch size (8), optimizer (AdamW), LR schedule (Cosine with warmup), and loss function.
- **Persistence**: Mounts Google Drive to preserve checkpoints across runtime disconnects and supports resuming.

## 1. Environment Verification & Google Drive Mount
Mount Google Drive so checkpoints and training logs are saved persistently.

In [ ]:
# If you uploaded ONLY this notebook file to Colab, this cell ensures the project files are present:
import os
if not os.path.exists('src') and not os.path.exists('config.yaml'):
    print('Cloning repository files into Colab environment...')
    # Replace with your GitHub repo URL if you pushed it:
    # !git clone https://github.com/<your-username>/<your-repo-name>.git .
    pass

# Mount Google Drive for persistent checkpoints
try:
    from google.colab import drive
    drive.mount('/content/drive')
    checkpoint_dir = '/content/drive/MyDrive/aeroscapes_checkpoints'
    import os
    os.makedirs(checkpoint_dir, exist_ok=True)
    print(f'Persistent checkpoint directory ready: {checkpoint_dir}')
except ImportError:
    print('Not running in Google Colab. Using local checkpoint directory.')

# Check GPU accelerator
import torch
print(f'PyTorch Version: {torch.__version__}')
print(f'CUDA Available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU Device: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')
else:
    print('WARNING: GPU not detected! Go to Runtime > Change runtime type > T4 GPU.')

## 2. Install Project Dependencies

In [ ]:
# Colab already has PyTorch, Torchvision, NumPy, Matplotlib, and SciPy pre-installed.
# We only need to install a few lightweight utilities directly without needing requirements.txt:
!pip install -q torchinfo gdown pytest pyyaml pillow
print('All necessary dependencies installed!')

## 3. Obtain AeroScapes Dataset
The dataset is sourced directly from the authors' repository (`ishann/aeroscapes`) linking Google Drive ID `1WmXcm0IamIA0QPpyxRfWKnicxZByA60v`.

> **Manual Fallback Steps (if automated download fails):**
> 1. Download `aeroscapes.zip` from https://drive.google.com/file/d/1WmXcm0IamIA0QPpyxRfWKnicxZByA60v/view?usp=sharing
> 2. Upload to your Google Drive as `MyDrive/aeroscapes.zip`
> 3. Run: `!unzip -q /content/drive/MyDrive/aeroscapes.zip -d Dataset`

In [ ]:
import os
if not os.path.exists('Dataset/train/img'):
    print('Downloading AeroScapes dataset archive...')
    # Try automated gdown
    exit_code = os.system('gdown --id 1WmXcm0IamIA0QPpyxRfWKnicxZByA60v -O aeroscapes.zip')
    if exit_code == 0 and os.path.exists('aeroscapes.zip'):
        print('Extracting dataset archive...')
        !unzip -q aeroscapes.zip -d Dataset
    elif os.path.exists('/content/drive/MyDrive/aeroscapes.zip'):
        print('Extracting from mounted Google Drive...')
        !unzip -q /content/drive/MyDrive/aeroscapes.zip -d Dataset
    else:
        print('ERROR: Automated download failed and archive not found in Drive.')
        print('Please follow the manual steps above.')
else:
    print('Dataset directory already present. Proceeding to inspection.')

## 4. Dataset Inspection, Mask Preprocessing & Splits Generation
- Verifies 1-to-1 pairing of 3,269 images and annotations with assertions.
- Decodes Supervisely JSON masks into indexed single-channel PNG masks (0..11).
- Generates fixed, seeded 70/15/15 splits (`splits/train.txt`, `splits/val.txt`, `splits/test.txt`).
- Computes and prints exact pixel frequencies across all classes.

In [ ]:
!python scripts/preprocess_dataset.py --raw_dir Dataset --mask_dir Dataset/preprocessed_masks --splits_dir splits

## 5. Run Pytest Unit Tests
Verifies hand-computed IoU metrics and geometric transform alignment.

In [ ]:
!pytest tests/

## 6. Preliminary Run: Unweighted vs. Class-Weighted Cross-Entropy Loss
Evaluates whether class weighting (inverse-sqrt frequency) or unweighted cross-entropy performs better on AeroScapes before running the full experiment.

In [ ]:
!python src/train.py --preliminary_loss_check --epochs 3

## 7. Train Model A (Baseline: No Skip Connections)
Compact 4-stage encoder-decoder with bottleneck only.

In [ ]:
!python src/train.py --model A --epochs 10 --seed 42

## 8. Train Model B (Controlled Change: Skip Fusion via Concatenation)
Identical to Model A in every hyperparameter, differing strictly by feature concatenation into matching decoder stages.

In [ ]:
!python src/train.py --model B --epochs 10 --seed 42

## 9. Standalone Evaluation & Latency Benchmarks
Evaluates both models on the held-out test split (491 images), benchmarks latency over 100 timed runs, and generates evidence outputs.

In [ ]:
!python src/evaluate.py --config config.yaml --out_dir results

## 10. Evidence Visualizations & Results Display
Display the qualitative comparison grids, failure analyses, boundary IoU comparisons, confusion matrices, and training curves.

In [ ]:
from IPython.display import Image, display, Markdown

print('=== Evaluation Metrics Summary Table ===')
with open('results/metrics.md', 'r') as f:
    display(Markdown(f.read()))

print('\n=== Qualitative Comparison Grid (Input / GT / Model A / Model B) ===')
display(Image('results/qualitative_grid.png'))

print('\n=== Failure Case Analysis: Lowest Per-Image mIoU ===')
display(Image('results/failure_cases_lowest_miou.png'))

print('\n=== Failure Analysis: Cases Where Model B Underperforms Model A ===')
display(Image('results/failure_cases_b_worse_than_a.png'))

print('\n=== Per-Class IoU Comparison (Model A vs Model B) ===')
display(Image('results/per_class_iou_comparison.png'))

print('\n=== Training Curves (Loss, mIoU, Learning Rate) ===')
display(Image('results/training_curves.png'))

print('\n=== Model Architectures Diagrams ===')
display(Image('results/Model_A_Baseline_diagram.png'))
display(Image('results/Model_B_Skips_diagram.png'))